In [1]:
# %% Cell 1 — imports & paths
import os, pickle
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

SRC = "/scratch/groups/horence/dcotter1/poppunk_lineage_pipeline/ef_clusters_refined"
DISTS_NPY = os.path.join(SRC, "ef_clusters_refined.refs.dists.npy")
DISTS_PKL = os.path.join(SRC, "ef_clusters_refined.refs.dists.pkl")
CLUSTERS_CSV = os.path.join(SRC, "ef_clusters_refined_clusters.csv")

OUTDIR = "."  # write only here
os.makedirs(OUTDIR, exist_ok=True)

# %% Cell 2 — inspect file structure first (nothing assumed)
with open(DISTS_PKL, "rb") as f:
    pkl = pickle.load(f)
print("PKL type:", type(pkl))
if hasattr(pkl, "__len__"):
    print("PKL len:", len(pkl))
    for i, x in enumerate(pkl):
        print(f"  elem {i}: type={type(x)}, "
              f"len={len(x) if hasattr(x,'__len__') else 'NA'}, "
              f"sample={x[:3] if hasattr(x,'__len__') else x}")

D = np.load(DISTS_NPY)
print("DISTS npy shape:", D.shape, "dtype:", D.dtype)

# %% Cell 3 — resolve sample names
# PopPUNK .dists.pkl is typically (rlist, qlist) where for a symmetric ref-ref
# distance set rlist == qlist == ordered sample names.
if isinstance(pkl, (tuple, list)) and len(pkl) >= 1:
    rlist = pkl[0]
    qlist = pkl[1] if len(pkl) > 1 else pkl[0]
else:
    raise ValueError("Unexpected pkl structure; inspect Cell 2 output.")

names = list(rlist)
n = len(names)
print("n samples:", n)
assert list(qlist) == names, "rlist != qlist; this may be a query-ref set, not symmetric."

# %% Cell 4 — build a square distance matrix (core distances)
# PopPUNK dists npy is usually shape (n_pairs, 2): columns = [core, accessory].
# Pairs are in upper-triangle (condensed) order matching scipy's squareform.
n_pairs = n * (n - 1) // 2
print("expected n_pairs (upper triangle):", n_pairs, "| npy rows:", D.shape[0])

assert D.ndim == 2 and D.shape[1] >= 1, "Expected 2D dists with >=1 column."
CORE_COL, ACC_COL = 0, 1  # adjust if inspection says otherwise

if D.shape[0] == n_pairs:
    core_condensed = D[:, CORE_COL].astype(float)
    core_sq = squareform(core_condensed)  # n x n, zero diagonal
elif D.shape[0] == n * n:
    core_sq = D[:, CORE_COL].reshape(n, n).astype(float)
    np.fill_diagonal(core_sq, 0.0)
else:
    raise ValueError(
        f"Cannot reconcile npy rows ({D.shape[0]}) with n={n}. "
        f"Inspect structure and adjust reshaping."
    )
print("core_sq shape:", core_sq.shape,
      "min/median/max:",
      float(core_sq.min()), float(np.median(core_sq)), float(core_sq.max()))

# %% Cell 5 — hierarchical clustering on core distances
core_condensed_final = squareform(core_sq, checks=False)  # ensure condensed form
Z = linkage(core_condensed_final, method="average")

# %% Cell 6 — cut the tree at several resolutions -> coarser clusters
# Choose thresholds based on the distance scale printed above. Tune as needed.
thresholds = [0.01, 0.02, 0.05, 0.10]

out = pd.DataFrame({"sample": names})

# original fine clusters (for comparison), read-only
try:
    fine = pd.read_csv(CLUSTERS_CSV)
    # PopPUNK cluster csv typically has columns like: Taxon, Cluster
    tcol = [c for c in fine.columns if c.lower() in ("taxon","sample","id","taxa")]
    ccol = [c for c in fine.columns if "cluster" in c.lower()]
    if tcol and ccol:
        fmap = dict(zip(fine[tcol[0]], fine[ccol[0]]))
        out["fine_cluster"] = out["sample"].map(fmap)
except Exception as e:
    print("Could not attach fine clusters:", e)

for t in thresholds:
    labels = fcluster(Z, t=t, criterion="distance")
    col = f"coarse_t{t}"
    out[col] = labels
    nclust = len(set(labels))
    sizes = pd.Series(labels).value_counts()
    n_small = int((sizes < 5).sum())
    print(f"t={t:<5} -> {nclust:>4} clusters | "
          f"singletons={int((sizes==1).sum())} | clusters<5={n_small}")

# %% Cell 7 — save output (only to current folder)
out_path = os.path.join(OUTDIR, "coarse_clusters.tsv")
out.to_csv(out_path, sep="\t", index=False)
print("Wrote:", os.path.abspath(out_path))
out.head()

# %% Cell 8 (optional) — pick one resolution and summarize cluster sizes
chosen = "coarse_t0.05"  # edit after seeing Cell 6 output
summary = (out.groupby(chosen)["sample"].count()
              .sort_values(ascending=False)
              .rename("n_samples").reset_index())
summary_path = os.path.join(OUTDIR, f"{chosen}_sizes.tsv")
summary.to_csv(summary_path, sep="\t", index=False)
print("Wrote:", os.path.abspath(summary_path))
summary.head(20)

PKL type: <class 'list'>
PKL len: 3
  elem 0: type=<class 'list'>, len=1188, sample=['ERR1007500', 'ERR1036024', 'ERR1036026']
  elem 1: type=<class 'list'>, len=1188, sample=['ERR1007500', 'ERR1036024', 'ERR1036026']
  elem 2: type=<class 'bool'>, len=NA, sample=True
DISTS npy shape: (705078, 2) dtype: float32
n samples: 1188
expected n_pairs (upper triangle): 705078 | npy rows: 705078
core_sq shape: (1188, 1188) min/median/max: 0.0 0.00485605001449585 0.04732269048690796
t=0.01  ->   15 clusters | singletons=7 | clusters<5=11
t=0.02  ->    3 clusters | singletons=1 | clusters<5=1
t=0.05  ->    1 clusters | singletons=0 | clusters<5=0
t=0.1   ->    1 clusters | singletons=0 | clusters<5=0
Wrote: /scratch/users/jiamuyu/proj_botryllus/flash/utility/coarse_clusters.tsv
Wrote: /scratch/users/jiamuyu/proj_botryllus/flash/utility/coarse_t0.05_sizes.tsv


,coarse_t0.05,n_samples
0,1,1188


In [1]:
# %% Standalone HDBSCAN clustering on PopPUNK core distances
# Read-only from source; writes only to current folder.

import os, pickle
import numpy as np
import pandas as pd
from scipy.spatial.distance import squareform
import hdbscan   # pip install hdbscan

# ---- paths (read-only) ----
SRC = "/scratch/groups/horence/dcotter1/poppunk_lineage_pipeline/ef_clusters_refined"
DISTS_NPY = os.path.join(SRC, "ef_clusters_refined.refs.dists.npy")
DISTS_PKL = os.path.join(SRC, "ef_clusters_refined.refs.dists.pkl")
CLUSTERS_CSV = os.path.join(SRC, "ef_clusters_refined_clusters.csv")
OUTDIR = "."

# ---- load sample names ----
with open(DISTS_PKL, "rb") as f:
    pkl = pickle.load(f)
rlist = pkl[0]
qlist = pkl[1] if isinstance(pkl, (tuple, list)) and len(pkl) > 1 else pkl[0]
names = list(rlist)
n = len(names)
assert list(qlist) == names, "rlist != qlist; not a symmetric ref-ref set."
print(f"n samples: {n}")

# ---- load distances and build square core-distance matrix ----
D = np.load(DISTS_NPY)
print("dists shape:", D.shape)
CORE_COL = 0  # column 0 = core, column 1 = accessory (adjust if needed)

n_pairs = n * (n - 1) // 2
if D.shape[0] == n_pairs:                     # condensed upper-triangle
    core_sq = squareform(D[:, CORE_COL].astype(float))
elif D.shape[0] == n * n:                     # full flattened
    core_sq = D[:, CORE_COL].reshape(n, n).astype(float)
    np.fill_diagonal(core_sq, 0.0)
else:
    raise ValueError(f"Can't reconcile npy rows ({D.shape[0]}) with n={n}.")

core_sq = np.ascontiguousarray(core_sq, dtype="float64")
core_sq = np.maximum(core_sq, core_sq.T)      # enforce symmetry
np.fill_diagonal(core_sq, 0.0)
print("core_sq:", core_sq.shape,
      "min/median/max:",
      float(core_sq.min()), float(np.median(core_sq)), float(core_sq.max()))

# ---- HDBSCAN ----
MIN_CLUSTER_SIZE = 5     # your "no clusters smaller than 5" target
MIN_SAMPLES = None       # None -> defaults to min_cluster_size; lower = fewer noise pts

clusterer = hdbscan.HDBSCAN(
    metric="precomputed",
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    cluster_selection_method="eom",   # 'eom' (default) or 'leaf' for finer clusters
)
labels = clusterer.fit_predict(core_sq)

# ---- summarize ----
lab = pd.Series(labels)
n_clusters = int((lab.unique() >= 0).sum())
n_noise = int((lab == -1).sum())
print(f"clusters: {n_clusters} | noise points (-1): {n_noise}")
print("cluster sizes:\n", lab[lab >= 0].value_counts().sort_index())

# ---- assemble output ----
out = pd.DataFrame({
    "sample": names,
    "hdbscan_cluster": labels,
    "membership_prob": clusterer.probabilities_,   # confidence per point
})

# attach original PopPUNK fine clusters for comparison (read-only)
try:
    fine = pd.read_csv(CLUSTERS_CSV)
    tcol = next(c for c in fine.columns if c.lower() in ("taxon","sample","id","taxa"))
    ccol = next(c for c in fine.columns if "cluster" in c.lower())
    out["fine_cluster"] = out["sample"].map(dict(zip(fine[tcol], fine[ccol])))
except Exception as e:
    print("Could not attach fine clusters:", e)

# ---- save (only to current folder) ----
out_path = os.path.join(OUTDIR, "hdbscan_clusters.tsv")
out.to_csv(out_path, sep="\t", index=False)
print("Wrote:", os.path.abspath(out_path))
out.head()

n samples: 1188
dists shape: (705078, 2)
core_sq: (1188, 1188) min/median/max: 0.0 0.00485605001449585 0.04732269048690796
clusters: 45 | noise points (-1): 494
cluster sizes:
 0      6
1     10
2     40
3     32
4      6
5     32
6     36
7     56
8     15
9     14
10    26
11    13
12     8
13     7
14     7
15     7
16    14
17    17
18     8
19    12
20     6
21    13
22     6
23     6
24    12
25    15
26    33
27    17
28    12
29     9
30    24
31     7
32    25
33     7
34     8
35    14
36     5
37    17
38    13
39    23
40    10
41    14
42     9
43    13
44    10
Name: count, dtype: int64
Wrote: /scratch/users/jiamuyu/proj_botryllus/flash/utility/hdbscan_clusters.tsv


,sample,hdbscan_cluster,membership_prob,fine_cluster
0,ERR1007500,-1,0.000000,117
1,ERR1036024,-1,0.000000,560
2,ERR1036026,7,0.889561,559
3,ERR1036033,-1,0.000000,558
4,ERR1036034,37,1.000000,557


In [2]:
from pathlib import Path
import pandas as pd

workdir = Path("/scratch/users/jiamuyu/proj_botryllus/splash2/260808_01_efaecium_strain")
cluster_file = workdir / "ef_clusters_refined_clusters.csv"
min_cluster_size = 5

clusters = pd.read_csv(cluster_file)
clusters = clusters.rename(columns={"Taxon": "sample_name", "Cluster": "cluster"})

cluster_sizes = clusters["cluster"].value_counts()
sparse_clusters = set(cluster_sizes[cluster_sizes < min_cluster_size].index)

clusters["cluster"] = clusters["cluster"].apply(
    lambda x: "sparse" if x in sparse_clusters else str(x)
)

for tsv in workdir.glob("*metadata.tsv"):
    df = pd.read_csv(tsv, sep="\t")
    
    out = df.merge(
        clusters[["sample_name", "cluster"]],
        on="sample_name",
        how="left"
    )
    out["cluster"] = out["cluster"].fillna("sparse")
    
    out_path = tsv.with_name(tsv.stem + ".with_cluster.tsv")
    out.to_csv(out_path, sep="\t", index=False)
    
    print(f"Wrote {out_path}")

Wrote /scratch/users/jiamuyu/proj_botryllus/splash2/260808_01_efaecium_strain/E_faecium_coreSplit_test_genomes_metadata.with_cluster.tsv
Wrote /scratch/users/jiamuyu/proj_botryllus/splash2/260808_01_efaecium_strain/E_faecium_test_genomes_metadata.with_cluster.tsv
Wrote /scratch/users/jiamuyu/proj_botryllus/splash2/260808_01_efaecium_strain/E_faecium_train_metadata.with_cluster.tsv
Wrote /scratch/users/jiamuyu/proj_botryllus/splash2/260808_01_efaecium_strain/E_faecium_coreSplit_train_metadata.with_cluster.tsv
